# Notebook 02 | Cross Strategy Execution Comparison

Compare TWAP, VWAP, POV, implementation shortfall, Almgren-Chriss, and the adaptive policy on a common synthetic market path. The emphasis is on schedule shape, execution quality, and cost decomposition under shared market conditions.

## Base Scenario and Shared Market Path

All strategies are evaluated on the same simulated intraday path so that differences in performance are attributable to the execution policy rather than path variation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from execution_engine.config import load_config
from execution_engine.market.price_process import IntradayMarketSimulator
from execution_engine.simulation.scenario_runner import run_strategy_suite
from execution_engine.visualization.plots_costs import plot_strategy_comparison
from execution_engine.visualization.plots_schedule import plot_execution_schedule, plot_inventory_trajectory

config = load_config(PROJECT_ROOT / 'configs' / 'base.yaml')
market = IntradayMarketSimulator(config).simulate(seed=11)
suite = run_strategy_suite(config=config, market=market, seed=11)
summary = suite.summary
summary[['rank', 'strategy', 'implementation_shortfall', 'implementation_shortfall_bps', 'completion_rate', 'average_participation']]

## Cross-Strategy Ranking

Compare implementation shortfall in basis points across the full strategy set. This is a single-path comparison and should be interpreted as a scenario-specific ranking, not a universal ordering.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_strategy_comparison(summary, metric='implementation_shortfall_bps', ax=ax)
plt.tight_layout()
plt.show()

## Schedule Shape and Inventory Trajectory

The next panel contrasts requested versus realized fills for representative strategies and shows how different policies manage the remaining inventory through time.

In [ ]:
results = {result.strategy_name: result for result in suite.results}
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
plot_execution_schedule(results['twap'], ax=axes[0, 0])
plot_execution_schedule(results['almgren_chriss'], ax=axes[0, 1])
plot_inventory_trajectory(results['implementation_shortfall'], ax=axes[1, 0])
plot_inventory_trajectory(results['adaptive_policy'], ax=axes[1, 1])
plt.tight_layout()
plt.show()

## Cost Decomposition Snapshot

Inspect the main cost drivers strategy by strategy. The table below is useful for separating favorable market drift from explicit trading frictions such as spread, temporary impact, and adverse selection.

In [ ]:
cost_table = pd.DataFrame(
    {
        result.strategy_name: result.cost_breakdown
        for result in suite.results
    }
).T
cost_table[['implementation_shortfall', 'market_drift', 'spread_paid', 'spread_capture', 'temporary_impact', 'permanent_impact', 'adverse_selection']]